# Contact Map Analysis — CESA1 / HVGlu3 Complex
**University of Jordan — CESA Project**

This notebook computes:
1. Overall contact frequency map — HVGlu3 ↔ CESA1 subunits 1 & 2 (all 6996 frames)
2. Per-cluster contact maps — how interface changes across conformational states
3. Cluster 3 special analysis — all three CESA1 subunits (tightest binding state)
4. Key binding residues — most persistent contacts

**Distance cutoff:** 4.5 Å (standard for protein-protein interfaces)

**Chain assignments:**
- Chain A = HVGlu3
- Chain B = CESA1 subunit 1
- Chain C = CESA1 subunit 2
- Chain D = CESA1 subunit 3 (Cluster 3 only)

**Note:** Run with `Python (md_analysis)` kernel

In [ ]:
import scipy
import MDAnalysis as mda
print(scipy.__version__)
print(mda.__version__)

In [ ]:
# ============================================================
# CELL 1 — Imports
# ============================================================
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warnings
warnings.filterwarnings('ignore')

import MDAnalysis as mda
from MDAnalysis.analysis import distances

plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['font.family']     = 'sans-serif'

print('MDAnalysis version:', mda.__version__)
print('All imports successful!')

In [ ]:
# ============================================================
# CELL 2 — File paths and parameters
# ============================================================

RUN_DIR    = Path('/Volumes/SAMSUNG/CESAproject/MD_after_docking/CESA1/result_CESA1_ClusPro/run/')
TOPOLOGY   = RUN_DIR / 'protein.pdb'
TRAJECTORY = RUN_DIR / 'CESA1_all_protein.xtc'
KMEANS_CSV = RUN_DIR / 'KMeans' / 'PCA_clusters.csv'
OUT_DIR    = RUN_DIR / 'ContactMap'
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Analysis parameters
CUTOFF = 4.5   # Å — standard for protein-protein interfaces

# Verify files
for f in [TOPOLOGY, TRAJECTORY, KMEANS_CSV]:
    status = '✓' if f.exists() else '✗ MISSING'
    size   = f'{f.stat().st_size / 1e6:.1f} MB' if f.exists() else ''
    print(f'  {status}  {f.name}  {size}')

print(f'\nCutoff       : {CUTOFF} Å')
print(f'Output folder: {OUT_DIR}')

In [ ]:
# ============================================================
# CELL 3 — Load trajectory and cluster assignments
# ============================================================

print('Loading trajectory (this may take a few minutes)...')
u = mda.Universe(str(TOPOLOGY), str(TRAJECTORY))
print(f'Trajectory loaded: {len(u.trajectory)} frames')
print(f'Total atoms: {len(u.atoms)}')

# Confirm chain structure
print('\nChain structure:')
for seg in u.segments:
    print(f'  Chain {seg.segid}: {len(seg.residues)} residues  '
          f'resid {seg.residues[0].resid}–{seg.residues[-1].resid}')

# Load cluster assignments from K-means
df_clusters = pd.read_csv(KMEANS_CSV)
print(f'\nCluster assignments loaded: {len(df_clusters)} frames')
print(df_clusters['Cluster'].value_counts().sort_index())

# Frame indices per cluster (0-based)
cluster_frames = {}
for c in sorted(df_clusters['Cluster'].unique()):
    cluster_frames[c] = df_clusters[df_clusters['Cluster'] == c].index.tolist()
    print(f'  Cluster {c+1}: {len(cluster_frames[c])} frames')

In [ ]:
# ============================================================
# CELL 4 — Core function: compute contact frequency
#
# For each frame in frame_list:
#   For each residue pair (HVGlu3 res, CESA1 res):
#     If minimum distance between any heavy atoms < cutoff → contact = 1
# Contact frequency = fraction of frames with contact
# ============================================================

def compute_contact_frequency(u, sel1_str, sel2_str, frame_list, cutoff=4.5):
    """
    Compute contact frequency matrix between two selections.
    
    Returns:
        contact_freq : 2D array (n_res1, n_res2) — fraction of frames in contact
        res1_ids     : residue IDs for selection 1
        res2_ids     : residue IDs for selection 2
        res1_names   : residue names for selection 1
        res2_names   : residue names for selection 2
    """
    sel1 = u.select_atoms(sel1_str)
    sel2 = u.select_atoms(sel2_str)

    res1_ids   = np.array([r.resid   for r in sel1.residues])
    res2_ids   = np.array([r.resid   for r in sel2.residues])
    res1_names = np.array([r.resname for r in sel1.residues])
    res2_names = np.array([r.resname for r in sel2.residues])
    res1_segs  = np.array([r.segment.segid for r in sel1.residues])
    res2_segs  = np.array([r.segment.segid for r in sel2.residues])

    n1 = len(sel1.residues)
    n2 = len(sel2.residues)
    contact_count = np.zeros((n1, n2), dtype=np.float32)

    n_frames = len(frame_list)
    report_every = max(1, n_frames // 10)

    for fi, frame_idx in enumerate(frame_list):
        u.trajectory[frame_idx]

        for i, res1 in enumerate(sel1.residues):
            atoms1 = res1.atoms
            pos1   = atoms1.positions

            for j, res2 in enumerate(sel2.residues):
                atoms2 = res2.atoms
                pos2   = atoms2.positions

                # Minimum distance between any atom pair
                dist_matrix = distances.distance_array(pos1, pos2)
                if dist_matrix.min() <= cutoff:
                    contact_count[i, j] += 1

        if (fi + 1) % report_every == 0:
            print(f'  Processed {fi+1}/{n_frames} frames ({(fi+1)/n_frames*100:.0f}%)')

    contact_freq = contact_count / n_frames

    return contact_freq, res1_ids, res2_ids, res1_names, res2_names, res1_segs, res2_segs

print('Contact frequency function defined!')
print(f'Will use cutoff: {CUTOFF} Å')

In [ ]:
# ============================================================
# CELL 5 — Plot function: contact frequency heatmap
# ============================================================

def plot_contact_map(contact_freq, res1_ids, res2_ids,
                     res1_segs, res2_segs,
                     title, output_path,
                     threshold=0.3):
    """
    Plot contact frequency heatmap.
    threshold: minimum frequency to display (filters noise)
    """
    # Mask contacts below threshold
    plot_data = contact_freq.copy()
    plot_data[plot_data < threshold] = 0

    fig, ax = plt.subplots(figsize=(14, 8))

    im = ax.imshow(
        plot_data.T,
        aspect='auto',
        origin='lower',
        cmap='YlOrRd',
        vmin=0, vmax=1,
        interpolation='nearest'
    )

    plt.colorbar(im, ax=ax, label='Contact frequency', fraction=0.02, pad=0.02)

    ax.set_xlabel('HVGlu3 residue index', fontsize=13, labelpad=8)
    ax.set_ylabel('CESA1 residue index', fontsize=13, labelpad=8)
    ax.set_title(title, fontsize=14, weight='bold', pad=15)

    # Add chain boundary lines for CESA1 (y-axis)
    unique_segs = []
    for s in res2_segs:
        if s not in unique_segs:
            unique_segs.append(s)

    cumulative = 0
    seg_centers = []
    seg_labels  = []
    chain_map   = {'B': 'CESA1 subunit 1', 'C': 'CESA1 subunit 2', 'D': 'CESA1 subunit 3'}

    for seg in unique_segs:
        seg_count = np.sum(res2_segs == seg)
        if cumulative > 0:
            ax.axhline(y=cumulative - 0.5, color='black',
                      linestyle='--', lw=1.0, alpha=0.5)
        seg_centers.append(cumulative + seg_count / 2)
        seg_labels.append(chain_map.get(seg, seg))
        cumulative += seg_count

    ax.set_yticks(seg_centers)
    ax.set_yticklabels(seg_labels, fontsize=10)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

    plt.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'Saved: {Path(output_path).name}')

print('Plot function defined!')

In [ ]:
# # ============================================================
# CELL 6 — OVERALL contact frequency map (LOAD OR COMPUTE)
# ============================================================

import os

npy_path = OUT_DIR / 'contact_freq_overall.npy'

# --- Reconstruct residue metadata (fast, from universe) ---
sel1 = u.select_atoms('segid A')
sel2 = u.select_atoms('segid B or segid C')
res1 = sel1.residues
res2 = sel2.residues
r1_ids   = res1.resids
r2_ids   = res2.resids
r1_names = [r.resname for r in res1]
r2_names = [r.resname for r in res2]
r1_segs  = [r.segid for r in res1]
r2_segs  = [r.segid for r in res2]

if npy_path.exists():
    # ✅ FAST PATH — load pre-computed data
    print(f'✅ Loading pre-computed contact matrix from:\n   {npy_path}')
    cf_overall = np.load(npy_path)
    print(f'   Loaded in seconds!\n')
else:
    # ⏳ SLOW PATH — compute from scratch
    print('Computing overall contact frequency map...')
    print('HVGlu3 (Chain A) ↔ CESA1 subunits 1+2 (Chains B+C)')
    print(f'Frames: all 6996 | Cutoff: {CUTOFF} Å')
    print('This will take a while — please wait...\n')

    all_frames = list(range(0, len(u.trajectory), 5))

    cf_overall, r1_ids, r2_ids, r1_names, r2_names, r1_segs, r2_segs = \
        compute_contact_frequency(
            u,
            sel1_str   = 'segid A',
            sel2_str   = 'segid B or segid C',
            frame_list = all_frames,
            cutoff     = CUTOFF
        )
    np.save(npy_path, cf_overall)
    print(f'\n✅ Saved to {npy_path}')

print(f'Contact matrix shape: {cf_overall.shape}')
print(f'Max contact frequency: {cf_overall.max():.3f}')
print(f'Contacts > 30%: {(cf_overall > 0.3).sum()}')
print(f'Contacts > 50%: {(cf_overall > 0.5).sum()}')
print(f'Contacts > 70%: {(cf_overall > 0.7).sum()}')

In [ ]:
# ============================================================
# CELL 7 — Contact map: dot matrix + top contact pairs bar chart
# ============================================================

def plot_contact_dot_and_bars(contact_freq, r1_ids, r2_ids,
                               r1_names, r2_names,
                               r1_segs, r2_segs,
                               title, output_path,
                               threshold=0.3, top_n=30):

    r1_ids   = np.asarray(r1_ids)
    r2_ids   = np.asarray(r2_ids)
    r1_names = np.asarray(r1_names)
    r2_names = np.asarray(r2_names)
    r1_segs  = np.asarray(r1_segs)
    r2_segs  = np.asarray(r2_segs)

    contact_mask = contact_freq >= threshold
    row_idx, col_idx = np.where(contact_mask)

    if len(row_idx) == 0:
        print('No contacts above threshold!')
        return

    # Get unique contact residues
    active_rows = np.unique(row_idx)
    active_cols = np.unique(col_idx)
    n_x = len(active_rows)
    n_y = len(active_cols)

    # Map from original index → plot position
    row_to_pos = {r: i for i, r in enumerate(active_rows)}
    col_to_pos = {c: i for i, c in enumerate(active_cols)}

    # Build scatter data
    x_pos = np.array([row_to_pos[r] for r in row_idx])
    y_pos = np.array([col_to_pos[c] for c in col_idx])
    freq  = contact_freq[row_idx, col_idx]

    # Labels
    x_labels = [f'{r1_names[i]}{r1_ids[i]}' for i in active_rows]
    y_labels = [f'{r2_names[j]}{r2_ids[j]}({r2_segs[j]})' for j in active_cols]

    # --- Build contact pair list for bar chart ---
    pairs = []
    for r, c in zip(row_idx, col_idx):
        label = f'{r1_names[r]}{r1_ids[r]} ↔ {r2_names[c]}{r2_ids[c]}({r2_segs[c]})'
        pairs.append((label, contact_freq[r, c]))
    pairs.sort(key=lambda x: x[1], reverse=True)
    top_pairs = pairs[:top_n]

    print(f'Contact residues: {n_x} (HVGlu3) × {n_y} (CESA1)')
    print(f'Total contact pairs ≥ {threshold}: {len(pairs)}')
    print(f'Showing top {len(top_pairs)} pairs in bar chart')

    # ===================== FIGURE =====================
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(20, max(8, n_y * 0.35 + 2)),
                                    gridspec_kw={'width_ratios': [3, 2], 'wspace': 0.4})

    # -------- LEFT: Dot matrix --------
    # Gridlines first (behind dots)
    for i in range(n_x):
        ax1.axvline(x=i, color='#e0e0e0', linewidth=0.5, zorder=0)
    for j in range(n_y):
        ax1.axhline(y=j, color='#e0e0e0', linewidth=0.5, zorder=0)

    # Scatter: circles at contact positions
    sc = ax1.scatter(
        x_pos, y_pos,
        c=freq,
        s=80,
        cmap='YlOrRd',
        vmin=threshold, vmax=1.0,
        edgecolors='#333333',
        linewidths=0.6,
        zorder=3,
        marker='o'
    )

    cbar = plt.colorbar(sc, ax=ax1, fraction=0.02, pad=0.02)
    cbar.set_label('Contact frequency', fontsize=10)

    ax1.set_xticks(np.arange(n_x))
    ax1.set_xticklabels(x_labels, fontsize=7.5, rotation=90, ha='center')
    ax1.set_yticks(np.arange(n_y))
    ax1.set_yticklabels(y_labels, fontsize=7.5)

    ax1.set_xlim(-0.5, n_x - 0.5)
    ax1.set_ylim(-0.5, n_y - 0.5)

    # Chain boundary lines
    chain_map = {'B': 'Sub1', 'C': 'Sub2', 'D': 'Sub3'}
    prev_seg = str(r2_segs[active_cols[0]])
    for k in range(1, n_y):
        curr_seg = str(r2_segs[active_cols[k]])
        if curr_seg != prev_seg:
            ax1.axhline(y=k - 0.5, color='black', linestyle='-', lw=2, zorder=2)
        prev_seg = curr_seg

    ax1.set_xlabel('HvGlu3 residue', fontsize=11, labelpad=8)
    ax1.set_ylabel('CESA1 residue', fontsize=11, labelpad=8)
    ax1.set_title('Contact Dot Matrix', fontsize=12, fontweight='bold')
    ax1.spines['top'].set_visible(False)
    ax1.spines['right'].set_visible(False)

    # -------- RIGHT: Top contact pairs bar chart --------
    pair_labels = [p[0] for p in top_pairs][::-1]
    pair_freqs  = [p[1] for p in top_pairs][::-1]

    colors = plt.cm.YlOrRd(np.linspace(0.3, 0.95, len(pair_freqs)))[::-1]

    bars = ax2.barh(
        np.arange(len(pair_labels)),
        pair_freqs,
        color=colors,
        edgecolor='#333333',
        linewidth=0.5,
        height=0.7
    )

    # Add frequency values at bar ends
    for i, (bar, freq_val) in enumerate(zip(bars, pair_freqs)):
        ax2.text(freq_val + 0.01, i, f'{freq_val:.2f}',
                 va='center', fontsize=7.5, fontweight='bold')

    ax2.set_yticks(np.arange(len(pair_labels)))
    ax2.set_yticklabels(pair_labels, fontsize=7.5)
    ax2.set_xlabel('Contact frequency', fontsize=11, labelpad=8)
    ax2.set_title(f'Top {len(top_pairs)} Contact Pairs', fontsize=12, fontweight='bold')
    ax2.set_xlim(0, 1.15)
    ax2.axvline(x=0.5, color='gray', linestyle='--', lw=0.8, alpha=0.5, label='50%')
    ax2.axvline(x=0.7, color='gray', linestyle=':', lw=0.8, alpha=0.5, label='70%')
    ax2.legend(fontsize=8, loc='lower right')
    ax2.spines['top'].set_visible(False)
    ax2.spines['right'].set_visible(False)

    fig.suptitle(title, fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    fig.savefig(output_path, dpi=300, bbox_inches='tight')
    plt.show()
    print(f'✅ Saved: {Path(output_path).name}')


# --- Run ---
plot_contact_dot_and_bars(
    cf_overall, r1_ids, r2_ids,
    r1_names, r2_names,
    r1_segs, r2_segs,
    title=(f'Interfacial Contacts — HvGlu3 ↔ CESA1 (cutoff {CUTOFF} Å, freq ≥ 30%)'),
    output_path=OUT_DIR / 'contact_map_overall.png',
    threshold=0.3,
    top_n=30
)


In [ ]:
# ============================================================
# CELL — Extract global variables from cf_overall
# Run this before the Fig. 1 cell
# ============================================================

sel1 = u.select_atoms('segid A')
sel2 = u.select_atoms('segid B or segid C')

res1 = sel1.residues
res2 = sel2.residues

r1_ids   = res1.resids
r2_ids   = res2.resids
r1_names = np.array([r.resname for r in res1])
r2_names = np.array([r.resname for r in res2])
r1_segs  = np.array([r.segid   for r in res1])
r2_segs  = np.array([r.segid   for r in res2])

print(f'r1 (HVGlu3):  {len(r1_ids)} residues')
print(f'r2 (CESA1):   {len(r2_ids)} residues')
print(f'cf_overall shape: {cf_overall.shape}')
print('✅ Ready for Fig. 1')

In [ ]:
# ============================================================
# CELL — Objectively identify the hydrophobic core residues
# From the OVERALL contact map (cf_overall)
# ============================================================

import pandas as pd

# ── Hydrophobic amino acids ──
HYDROPHOBIC = {'ALA', 'VAL', 'ILE', 'LEU', 'MET', 'PHE', 'TRP', 'PRO', 'TYR'}

FREQ_THRESHOLD = 0.85   # ≥ 85% = dominant persistent contacts

# ── Build full contact pair dataframe from cf_overall ──
rows = []
for i in range(len(r1_ids)):
    for j in range(len(r2_ids)):
        freq = cf_overall[i, j]
        if freq >= FREQ_THRESHOLD:
            rows.append({
                'HVGlu3_resid'  : r1_ids[i],
                'HVGlu3_resname': r1_names[i],
                'CESA1_chain'   : r2_segs[j],
                'CESA1_resid'   : r2_ids[j],
                'CESA1_resname' : r2_names[j],
                'frequency'     : round(freq, 3)
            })

df_overall = pd.DataFrame(rows).sort_values('frequency', ascending=False)
print(f'Total contacts ≥ {FREQ_THRESHOLD*100:.0f}%: {len(df_overall)} pairs\n')

# ── Filter: both residues hydrophobic ──
df_hydro = df_overall[
    (df_overall['HVGlu3_resname'].isin(HYDROPHOBIC)) &
    (df_overall['CESA1_resname'].isin(HYDROPHOBIC))
].copy()

print(f'Hydrophobic core pairs: {len(df_hydro)}\n')
print(df_hydro[['HVGlu3_resname','HVGlu3_resid',
                'CESA1_chain','CESA1_resid','CESA1_resname',
                'frequency']].to_string(index=False))

# ── HVGlu3 core residues ranked ──
print('\n── HVGlu3 hydrophobic core residues ──')
hvglu3_core = df_hydro.groupby(['HVGlu3_resname','HVGlu3_resid'])['frequency'].agg(
    n_contacts='count', max_freq='max', mean_freq='mean'
).sort_values('max_freq', ascending=False)
print(hvglu3_core.to_string())

# ── CESA1 core residues ranked ──
print('\n── CESA1 hydrophobic core residues ──')
cesa1_core = df_hydro.groupby(['CESA1_chain','CESA1_resname','CESA1_resid'])['frequency'].agg(
    n_contacts='count', max_freq='max', mean_freq='mean'
).sort_values('max_freq', ascending=False)
print(cesa1_core.to_string())

# ── Save ──
df_overall.to_csv(OUT_DIR / 'contacts_overall_freq85.csv', index=False)
df_hydro.to_csv(OUT_DIR / 'hydrophobic_core_overall.csv', index=False)
print('\n✅ Saved both CSVs')

In [ ]:
# ============================================================
# CELL — Objectively identify Chain C interaction zone
# From the OVERALL contact map (cf_overall)
# ============================================================

# ── Filter: only HVGlu3 ↔ Chain C contacts ──
df_chainC = df_overall[df_overall['CESA1_chain'] == 'C'].copy()

print(f'Total chain C contacts ≥ 85%: {len(df_chainC)} pairs\n')
print(df_chainC[['HVGlu3_resname','HVGlu3_resid',
                 'CESA1_chain','CESA1_resid','CESA1_resname',
                 'frequency']].to_string(index=False))

# ── Now lower threshold to 50% to see full chain C picture ──
rows_50 = []
for i in range(len(r1_ids)):
    for j in range(len(r2_ids)):
        freq = cf_overall[i, j]
        if freq >= 0.50 and r2_segs[j] == 'C':
            rows_50.append({
                'HVGlu3_resid'  : r1_ids[i],
                'HVGlu3_resname': r1_names[i],
                'CESA1_chain'   : r2_segs[j],
                'CESA1_resid'   : r2_ids[j],
                'CESA1_resname' : r2_names[j],
                'frequency'     : round(cf_overall[i, j], 3)
            })

df_C50 = pd.DataFrame(rows_50).sort_values('frequency', ascending=False)

print(f'\nChain C contacts ≥ 50%: {len(df_C50)} pairs\n')
print(df_C50[['HVGlu3_resname','HVGlu3_resid',
              'CESA1_resid','CESA1_resname',
              'frequency']].to_string(index=False))

# ── Summary by HVGlu3 residue ──
print('\n── HVGlu3 residues contacting chain C (≥50%) ──')
summary = df_C50.groupby(['HVGlu3_resname','HVGlu3_resid'])['frequency'].agg(
    n_contacts='count', max_freq='max', mean_freq='mean'
).sort_values('max_freq', ascending=False)
print(summary.to_string())

# ── Summary by CESA1 chain C residue ──
print('\n── Chain C residues contacted (≥50%) ──')
cesa1_C_summary = df_C50.groupby(['CESA1_resname','CESA1_resid'])['frequency'].agg(
    n_contacts='count', max_freq='max', mean_freq='mean'
).sort_values('max_freq', ascending=False)
print(cesa1_C_summary.to_string())

# ── Save ──
df_C50.to_csv(OUT_DIR / 'contacts_chainC_overall_freq50.csv', index=False)
print('\n✅ Saved chain C contacts CSV')

In [ ]:
# ============================================================
# CELL — Chain C contacts split into primary and additional
# ============================================================

THRESHOLD_PRIMARY    = 0.85   # primary / dominant contacts
THRESHOLD_ADDITIONAL = 0.50   # additional contacts

# ── Primary contacts (≥ 85%) ──
df_C_primary = df_C50[df_C50['frequency'] >= THRESHOLD_PRIMARY].copy()

# ── Additional contacts (≥ 50% BUT < 85%) — exclude primary ──
df_C_additional = df_C50[df_C50['frequency'] < THRESHOLD_PRIMARY].copy()

print('── PRIMARY contacts (≥ 85%) ──')
print(df_C_primary[['HVGlu3_resname','HVGlu3_resid',
                     'CESA1_resid','CESA1_resname',
                     'frequency']].to_string(index=False))

print(f'\n── ADDITIONAL contacts (50–85%) ──')
print(df_C_additional[['HVGlu3_resname','HVGlu3_resid',
                        'CESA1_resid','CESA1_resname',
                        'frequency']].to_string(index=False))

In [ ]:
# Chain D contacts split into primary and additional
df_D50 = pd.DataFrame([
    {'HVGlu3_resid': r1_ids[i], 'HVGlu3_resname': r1_names[i],
     'CESA1_chain': r2_segs[j], 'CESA1_resid': r2_ids[j],
     'CESA1_resname': r2_names[j], 'frequency': round(cf_overall[i,j], 3)}
    for i in range(len(r1_ids))
    for j in range(len(r2_ids))
    if cf_overall[i,j] >= 0.50 and r2_segs[j] == 'D'
]).sort_values('frequency', ascending=False)

df_D_primary    = df_D50[df_D50['frequency'] >= 0.85]
df_D_additional = df_D50[df_D50['frequency'] <  0.85]

print('── Chain D PRIMARY (≥85%) ──')
print(df_D_primary[['HVGlu3_resname','HVGlu3_resid','CESA1_resid','CESA1_resname','frequency']].to_string(index=False))
print('\n── Chain D ADDITIONAL (50–85%) ──')
print(df_D_additional[['HVGlu3_resname','HVGlu3_resid','CESA1_resid','CESA1_resname','frequency']].to_string(index=False))

In [ ]:
print(f'Saved at: {OUT_DIR / "contact_map_overall.png"}')
print(f'File exists: {(OUT_DIR / "contact_map_overall.png").exists()}')


In [ ]:
# ============================================================
# CELL 8 — Load per-cluster contact maps from disk (NO recomputation)
# ============================================================

cf_clusters = {}

for cluster_num in [1, 2, 3, 4]:
    npy_path = OUT_DIR / f'contact_freq_cluster{cluster_num}.npy'

    # Cluster 3 uses all three CESA1 subunits
    if cluster_num == 3:
        cesa1_sel = 'segid B or segid C or segid D'
        label     = 'CESA1 subunits 1+2+3'
    else:
        cesa1_sel = 'segid B or segid C'
        label     = 'CESA1 subunits 1+2'

    # Reconstruct residue metadata (fast)
    sel1 = u.select_atoms('segid A')
    sel2 = u.select_atoms(cesa1_sel)
    res1 = sel1.residues
    res2 = sel2.residues

    if npy_path.exists():
        print(f'✅ Cluster {cluster_num}: Loading from {npy_path.name}')
        cf = np.load(npy_path)
    else:
        print(f'⚠️  Cluster {cluster_num}: {npy_path.name} NOT FOUND — skipping!')
        continue

    # Get frame count from cluster_frames if available
    n_frames = len(cluster_frames.get(cluster_num - 1, [])) if 'cluster_frames' in dir() else '?'

    cf_clusters[cluster_num] = {
        'cf'        : cf,
        'r1_ids'    : res1.resids,
        'r2_ids'    : res2.resids,
        'r1_names'  : np.array([r.resname for r in res1]),
        'r2_names'  : np.array([r.resname for r in res2]),
        'r1_segs'   : np.array([r.segid for r in res1]),
        'r2_segs'   : np.array([r.segid for r in res2]),
        'n_frames'  : n_frames
    }

    print(f'   Shape: {cf.shape} | Max freq: {cf.max():.3f} | '
          f'>30%: {(cf > 0.3).sum()} | >50%: {(cf > 0.5).sum()}')

print(f'\n✅ Loaded {len(cf_clusters)} cluster contact maps from disk!')


In [ ]:
# ============================================================
# CELL — Extract all contacts >30% per cluster, then find
# HvGLU3 residues common to all 4 clusters
# ============================================================

THRESHOLD = 0.30
cluster_contact_dfs = {}

for cluster_num, data in cf_clusters.items():
    cf       = data['cf']
    r1_ids   = data['r1_ids']
    r1_names = data['r1_names']
    r2_ids   = data['r2_ids']
    r2_names = data['r2_names']
    r2_segs  = data['r2_segs']

    rows = []
    idx_i, idx_j = np.where(cf > THRESHOLD)
    for i, j in zip(idx_i, idx_j):
        rows.append({
            'HVGlu3_resid'  : r1_ids[i],
            'HVGlu3_resname': r1_names[i],
            'CESA1_chain'   : r2_segs[j],
            'CESA1_resid'   : r2_ids[j],
            'CESA1_resname' : r2_names[j],
            'frequency'     : round(cf[i, j], 3)
        })

    df = pd.DataFrame(rows).sort_values('frequency', ascending=False)
    cluster_contact_dfs[cluster_num] = df
    print(f'Cluster {cluster_num}: {len(df)} contacts >{int(THRESHOLD*100)}%')

# ── Save each cluster's contact list ──
for cluster_num, df in cluster_contact_dfs.items():
    df.to_csv(OUT_DIR / f'contacts_cluster{cluster_num}_freq30.csv', index=False)

# ── Find HvGLU3 residues present in ALL 4 clusters at >30% ──
resid_sets = {
    cluster_num: set(df['HVGlu3_resid'].unique())
    for cluster_num, df in cluster_contact_dfs.items()
}

common_resids = set.intersection(*resid_sets.values())
print(f'\nHvGLU3 residues with contacts >30% in ALL 4 clusters: {len(common_resids)}')
print(sorted(common_resids))

# ── Build a readable summary table for these common residues ──
summary_rows = []
for resid in sorted(common_resids):
    row = {'HVGlu3_resid': resid}
    for cluster_num, df in cluster_contact_dfs.items():
        sub = df[df['HVGlu3_resid'] == resid]
        row[f'max_freq_cluster{cluster_num}'] = sub['frequency'].max()
        row[f'n_contacts_cluster{cluster_num}'] = len(sub)
        row['resname'] = sub['HVGlu3_resname'].iloc[0] if len(sub) else None
    summary_rows.append(row)

df_common = pd.DataFrame(summary_rows)
print('\n── Summary: HvGLU3 residues conserved across all clusters (>30%) ──')
print(df_common.to_string(index=False))

df_common.to_csv(OUT_DIR / 'hvglu3_residues_common_all_clusters_30pct.csv', index=False)
print('\n✅ Saved common-residue summary CSV')

In [ ]:
# ============================================================
# CELL — Check Arg67 specifically across all clusters and overall
# ============================================================

TARGET_RESID = 67

print(f'=== Checking HvGLU3 residue {TARGET_RESID} (expected: Arg67) ===\n')

# 1) Confirm what residue 67 actually is in the structure
idx67 = np.where(r1_ids == TARGET_RESID)[0]
if len(idx67) == 0:
    print(f'❌ Residue {TARGET_RESID} not found in r1_ids at all — check residue numbering/offset')
else:
    print(f'✅ Residue {TARGET_RESID} found: resname = {r1_names[idx67[0]]}')

# 2) Check OVERALL contact map — any contact at ANY frequency (not just >50%)
if len(idx67) > 0:
    i = idx67[0]
    row_overall = cf_overall[i, :]
    max_overall = row_overall.max()
    print(f'\nOverall ensemble — max contact frequency for resid {TARGET_RESID}: {max_overall:.4f}')
    if max_overall > 0:
        top_j = row_overall.argmax()
        print(f'   Highest-frequency partner: {r2_names[top_j]}{r2_ids[top_j]} (chain {r2_segs[top_j]}) at {max_overall:.4f}')
    # show all nonzero contacts, sorted
    nonzero = np.where(row_overall > 0)[0]
    print(f'   Total nonzero contacts: {len(nonzero)}')
    for j in sorted(nonzero, key=lambda x: -row_overall[x])[:10]:
        print(f'     {r2_names[j]}{r2_ids[j]} (chain {r2_segs[j]}): {row_overall[j]:.4f}')

# 3) Check EACH CLUSTER — same residue
print(f'\n=== Per-cluster check for resid {TARGET_RESID} ===')
for cluster_num, data in cf_clusters.items():
    cf       = data['cf']
    r1_ids_c = data['r1_ids']
    r2_ids_c = data['r2_ids']
    r2_names_c = data['r2_names']
    r2_segs_c  = data['r2_segs']

    idx_c = np.where(r1_ids_c == TARGET_RESID)[0]
    if len(idx_c) == 0:
        print(f'Cluster {cluster_num}: residue {TARGET_RESID} not in r1_ids_c')
        continue
    i = idx_c[0]
    row = cf[i, :]
    max_freq = row.max()
    print(f'Cluster {cluster_num}: max frequency = {max_freq:.4f}', end='')
    if max_freq > 0:
        top_j = row.argmax()
        print(f'  | top partner: {r2_names_c[top_j]}{r2_ids_c[top_j]} (chain {r2_segs_c[top_j]})')
    else:
        print('  | ZERO contact across entire trajectory in this cluster')

In [ ]:
# ============================================================
# CELL — Check Trp75 specifically across all clusters and overall
# ============================================================

TARGET_RESID = 75

print(f'=== Checking HvGLU3 residue {TARGET_RESID} (expected: Trp75) ===\n')

idx75 = np.where(r1_ids == TARGET_RESID)[0]
if len(idx75) == 0:
    print(f'❌ Residue {TARGET_RESID} not found in r1_ids at all — check residue numbering/offset')
else:
    print(f'✅ Residue {TARGET_RESID} found: resname = {r1_names[idx75[0]]}')

if len(idx75) > 0:
    i = idx75[0]
    row_overall = cf_overall[i, :]
    max_overall = row_overall.max()
    print(f'\nOverall ensemble — max contact frequency for resid {TARGET_RESID}: {max_overall:.4f}')
    if max_overall > 0:
        top_j = row_overall.argmax()
        print(f'   Highest-frequency partner: {r2_names[top_j]}{r2_ids[top_j]} (chain {r2_segs[top_j]}) at {max_overall:.4f}')
    nonzero = np.where(row_overall > 0)[0]
    print(f'   Total nonzero contacts: {len(nonzero)}')
    for j in sorted(nonzero, key=lambda x: -row_overall[x])[:10]:
        print(f'     {r2_names[j]}{r2_ids[j]} (chain {r2_segs[j]}): {row_overall[j]:.4f}')

print(f'\n=== Per-cluster check for resid {TARGET_RESID} ===')
for cluster_num, data in cf_clusters.items():
    cf       = data['cf']
    r1_ids_c = data['r1_ids']
    r2_ids_c = data['r2_ids']
    r2_names_c = data['r2_names']
    r2_segs_c  = data['r2_segs']

    idx_c = np.where(r1_ids_c == TARGET_RESID)[0]
    if len(idx_c) == 0:
        print(f'Cluster {cluster_num}: residue {TARGET_RESID} not in r1_ids_c')
        continue
    i = idx_c[0]
    row = cf[i, :]
    max_freq = row.max()
    print(f'Cluster {cluster_num}: max frequency = {max_freq:.4f}', end='')
    if max_freq > 0:
        top_j = row.argmax()
        print(f'  | top partner: {r2_names_c[top_j]}{r2_ids_c[top_j]} (chain {r2_segs_c[top_j]})')
    else:
        print('  | ZERO contact across entire trajectory in this cluster')

In [ ]:
# ============================================================
# CELL 9 — Plot per-cluster contact maps (dot matrix + bar chart)
# ============================================================

cluster_populations = {1: 24.5, 2: 27.9, 3: 12.8, 4: 34.8}

for cluster_num, data in cf_clusters.items():
    pop = cluster_populations[cluster_num]

    if cluster_num == 3:
        subunit_label = 'Subunits 1+2+3'
    else:
        subunit_label = 'Subunits 1+2'

    plot_contact_dot_and_bars(
        data['cf'],
        data['r1_ids'], data['r2_ids'],
        data['r1_names'], data['r2_names'],
        data['r1_segs'], data['r2_segs'],
        title       = f'Cluster {cluster_num} ({pop}%) — '
                      f'HvGlu3 ↔ CESA1 {subunit_label}\n'
                      f'({data["n_frames"]} frames, cutoff {CUTOFF} Å, freq ≥ 30%)',
        output_path = OUT_DIR / f'contact_map_cluster{cluster_num}.png',
        threshold   = 0.3,
        top_n       = 30
    )


In [ ]:
# ============================================================
# CELL 10 — Combined figure: all clusters (A, B, C, D)
# ============================================================

cluster_populations = {1: 24.5, 2: 27.9, 3: 12.8, 4: 34.8}
panel_labels = {1: 'A', 2: 'B', 3: 'C', 4: 'D'}

# --- Determine figure size based on max contact residues ---
max_ny = 0
for data in cf_clusters.values():
    cf = data['cf']
    active_cols = np.unique(np.where(cf >= 0.3)[1])
    max_ny = max(max_ny, len(active_cols))

fig, axes = plt.subplots(2, 2, figsize=(24, 22))
axes = axes.flatten()

for idx, (cluster_num, data) in enumerate(sorted(cf_clusters.items())):
    ax = axes[idx]
    pop = cluster_populations[cluster_num]

    cf = data['cf']
    r1_ids_c   = np.asarray(data['r1_ids'])
    r2_ids_c   = np.asarray(data['r2_ids'])
    r1_names_c = np.asarray(data['r1_names'])
    r2_names_c = np.asarray(data['r2_names'])
    r1_segs_c  = np.asarray(data['r1_segs'])
    r2_segs_c  = np.asarray(data['r2_segs'])

    if cluster_num == 3:
        subunit_label = 'Sub 1+2+3'
    else:
        subunit_label = 'Sub 1+2'

    # Find contact residues
    contact_mask = cf >= 0.3
    row_idx, col_idx = np.where(contact_mask)

    if len(row_idx) == 0:
        ax.text(0.5, 0.5, 'No contacts ≥ 30%', ha='center', va='center',
                transform=ax.transAxes, fontsize=14)
        ax.set_title(f'Cluster {cluster_num} ({pop}%)')
        continue

    active_rows = np.unique(row_idx)
    active_cols = np.unique(col_idx)
    n_x = len(active_rows)
    n_y = len(active_cols)

    row_to_pos = {r: i for i, r in enumerate(active_rows)}
    col_to_pos = {c: i for i, c in enumerate(active_cols)}

    x_pos = np.array([row_to_pos[r] for r in row_idx])
    y_pos = np.array([col_to_pos[c] for c in col_idx])
    freq  = cf[row_idx, col_idx]

    x_labels = [f'{r1_names_c[i]}{r1_ids_c[i]}' for i in active_rows]
    y_labels = [f'{r2_names_c[j]}{r2_ids_c[j]}({r2_segs_c[j]})' for j in active_cols]

    # Gridlines
    for i in range(n_x):
        ax.axvline(x=i, color='#e8e8e8', linewidth=0.4, zorder=0)
    for j in range(n_y):
        ax.axhline(y=j, color='#e8e8e8', linewidth=0.4, zorder=0)

    # Scatter dots
    sc = ax.scatter(
        x_pos, y_pos,
        c=freq,
        s=60,
        cmap='YlOrRd',
        vmin=0.3, vmax=1.0,
        edgecolors='#333333',
        linewidths=0.5,
        zorder=3,
        marker='o'
    )

    ax.set_xticks(np.arange(n_x))
    ax.set_xticklabels(x_labels, fontsize=6, rotation=90, ha='center')
    ax.set_yticks(np.arange(n_y))
    ax.set_yticklabels(y_labels, fontsize=6)
    ax.set_xlim(-0.5, n_x - 0.5)
    ax.set_ylim(-0.5, n_y - 0.5)

    # Chain boundaries
    chain_map = {'B': 'Sub1', 'C': 'Sub2', 'D': 'Sub3'}
    prev_seg = str(r2_segs_c[active_cols[0]])
    for k in range(1, n_y):
        curr_seg = str(r2_segs_c[active_cols[k]])
        if curr_seg != prev_seg:
            ax.axhline(y=k - 0.5, color='black', linestyle='-', lw=1.5, zorder=2)
        prev_seg = curr_seg

    ax.set_xlabel('HvGlu3 residue', fontsize=9, labelpad=5)
    ax.set_ylabel('CESA1 residue', fontsize=9, labelpad=5)

    # Panel label (A, B, C, D) — bold, top-left corner
    ax.text(-0.08, 1.05, panel_labels[cluster_num],
            transform=ax.transAxes, fontsize=20, fontweight='bold',
            va='top', ha='right')

    ax.set_title(f'Cluster {cluster_num} ({pop}%) — HvGlu3 ↔ CESA1 {subunit_label}\n'
                 f'({data["n_frames"]} frames, {len(row_idx)} contacts ≥ 30%)',
                 fontsize=10, fontweight='bold', pad=10)

    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

# Shared colorbar
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
sm = plt.cm.ScalarMappable(cmap='YlOrRd', norm=plt.Normalize(vmin=0.3, vmax=1.0))
sm.set_array([])
cbar = fig.colorbar(sm, cax=cbar_ax)
cbar.set_label('Contact frequency', fontsize=11)

plt.subplots_adjust(hspace=0.45, wspace=0.35, right=0.90)
fig.savefig(OUT_DIR / 'contact_map_all_clusters_combined.png', dpi=300, bbox_inches='tight')
fig.savefig(OUT_DIR / 'contact_map_all_clusters_combined.pdf', dpi=300, bbox_inches='tight')
plt.show()
print(f'✅ Saved: contact_map_all_clusters_combined.png + .pdf')


In [ ]:
# ============================================================
# CELL 10 — Key binding residues
#            Extract most persistent contacts (frequency > 50%)
#            from overall map and Cluster 3
# ============================================================

def extract_key_contacts(cf, r1_ids, r2_ids, r1_names, r2_names,
                          r1_segs, r2_segs, threshold=0.5, label=''):
    print(f'\n=== Key contacts ({label}) — frequency > {threshold*100:.0f}% ===')

    rows = []
    for i in range(cf.shape[0]):
        for j in range(cf.shape[1]):
            if cf[i, j] >= threshold:
                rows.append({
                    'HVGlu3_resid'   : r1_ids[i],
                    'HVGlu3_resname' : r1_names[i],
                    'CESA1_chain'    : r2_segs[j],
                    'CESA1_resid'    : r2_ids[j],
                    'CESA1_resname'  : r2_names[j],
                    'frequency'      : round(cf[i, j], 3)
                })

    if not rows:
        print('  No contacts above threshold')
        return pd.DataFrame()

    df = pd.DataFrame(rows).sort_values('frequency', ascending=False)
    print(f'  Total contacts: {len(df)}')
    print(df.head(20).to_string(index=False))
    return df

# Overall key contacts
df_overall_contacts = extract_key_contacts(
    cf_overall, r1_ids, r2_ids, r1_names, r2_names,
    r1_segs, r2_segs,
    threshold=0.5, label='Overall'
)
df_overall_contacts.to_csv(OUT_DIR / 'key_contacts_overall.csv', index=False)

# Cluster 3 key contacts (tightest binding state)
c3 = cf_clusters[3]
df_c3_contacts = extract_key_contacts(
    c3['cf'], c3['r1_ids'], c3['r2_ids'],
    c3['r1_names'], c3['r2_names'],
    c3['r1_segs'], c3['r2_segs'],
    threshold=0.5, label='Cluster 3 (tightest)'
)
df_c3_contacts.to_csv(OUT_DIR / 'key_contacts_cluster3.csv', index=False)

print('\nKey contact files saved!')

In [ ]:
# ============================================================
# CELL 11 — Summary: most contacted residues per protein
# ============================================================

if len(df_overall_contacts) > 0:
    print('=== Most contacted HVGlu3 residues (overall) ===')
    hvglu3_contacts = df_overall_contacts.groupby(
        ['HVGlu3_resid', 'HVGlu3_resname']
    )['frequency'].max().reset_index()
    hvglu3_contacts = hvglu3_contacts.sort_values('frequency', ascending=False)
    print(hvglu3_contacts.head(15).to_string(index=False))

    print('\n=== Most contacted CESA1 residues (overall) ===')
    cesa1_contacts = df_overall_contacts.groupby(
        ['CESA1_chain', 'CESA1_resid', 'CESA1_resname']
    )['frequency'].max().reset_index()
    cesa1_contacts = cesa1_contacts.sort_values('frequency', ascending=False)
    print(cesa1_contacts.head(15).to_string(index=False))

    # Save summaries
    hvglu3_contacts.to_csv(OUT_DIR / 'hvglu3_key_residues.csv', index=False)
    cesa1_contacts.to_csv(OUT_DIR  / 'cesa1_key_residues.csv',  index=False)
    print('\nSummary files saved!')

In [ ]:
# ============================================================
# CELL for publication — Fig. 5: Combined dot plot (1a) + bar chart (1b)
# Publication-quality, white background, journal-ready
# ============================================================

from matplotlib import rcParams
from matplotlib.lines import Line2D
import matplotlib.gridspec as gridspec

rcParams['font.family']  = 'Arial'
rcParams['pdf.fonttype'] = 42   # editable fonts in Illustrator

THRESHOLD = 0.5
TOP_N     = 20   # top N pairs in bar chart — adjust as needed

# ── Prepare data ──
r1_ids   = np.asarray(r1_ids)
r2_ids   = np.asarray(r2_ids)
r1_names = np.asarray(r1_names)
r2_names = np.asarray(r2_names)
r1_segs  = np.asarray(r1_segs)
r2_segs  = np.asarray(r2_segs)

contact_mask       = cf_overall >= THRESHOLD
row_idx, col_idx   = np.where(contact_mask)
active_rows        = np.unique(row_idx)
active_cols        = np.unique(col_idx)
n_x                = len(active_rows)
n_y                = len(active_cols)
row_to_pos         = {r: i for i, r in enumerate(active_rows)}
col_to_pos         = {c: i for i, c in enumerate(active_cols)}
x_pos              = np.array([row_to_pos[r] for r in row_idx])
y_pos              = np.array([col_to_pos[c] for c in col_idx])
freq               = cf_overall[row_idx, col_idx]

x_labels = [f'{r1_names[i]}{r1_ids[i]}'           for i in active_rows]
y_labels = [f'{r2_names[j]}{r2_ids[j]} ({r2_segs[j]})' for j in active_cols]

# Build sorted pair list
pairs = []
for r, c in zip(row_idx, col_idx):
    label = f'{r1_names[r]}{r1_ids[r]} ↔ {r2_names[c]}{r2_ids[c]} ({r2_segs[c]})'
    pairs.append((label, cf_overall[r, c]))
pairs.sort(key=lambda x: x[1], reverse=True)
top_pairs   = pairs[:TOP_N]
pair_labels = [p[0] for p in top_pairs][::-1]
pair_freqs  = [p[1] for p in top_pairs][::-1]

# ── Figure layout ──
fig = plt.figure(figsize=(22, max(10, n_y * 0.32 + 3)))
fig.patch.set_facecolor('white')

gs = gridspec.GridSpec(
    1, 2,
    width_ratios=[2.8, 1.8],
    wspace=0.45,
    left=0.06, right=0.97,
    top=0.92,  bottom=0.12
)

ax1 = fig.add_subplot(gs[0])   # Fig. 1a — dot plot
ax2 = fig.add_subplot(gs[1])   # Fig. 1b — bar chart

# ══════════════════════════════════════════════
# Fig. 1a — Contact Dot Matrix
# ══════════════════════════════════════════════
ax1.set_facecolor('white')

# Light gridlines
for i in range(n_x):
    ax1.axvline(x=i, color='#eeeeee', linewidth=0.4, zorder=0)
for j in range(n_y):
    ax1.axhline(y=j, color='#eeeeee', linewidth=0.4, zorder=0)

# Dots
sc = ax1.scatter(
    x_pos, y_pos,
    c=freq,
    s=70,
    cmap='YlOrRd',
    vmin=THRESHOLD, vmax=1.0,
    edgecolors='#444444',
    linewidths=0.5,
    zorder=3,
    marker='o'
)

cbar = plt.colorbar(sc, ax=ax1, fraction=0.025, pad=0.02)
cbar.set_label('Contact frequency', fontsize=10)
cbar.ax.tick_params(labelsize=9)

# Axis ticks
ax1.set_xticks(np.arange(n_x))
ax1.set_xticklabels(x_labels, fontsize=7, rotation=90, ha='center')
ax1.set_yticks(np.arange(n_y))
ax1.set_yticklabels(y_labels, fontsize=7.5)
ax1.set_xlim(-0.5, n_x - 0.5)
ax1.set_ylim(-0.5, n_y - 0.5)

# Chain boundary lines (between CESA1 subunits)
prev_seg = str(r2_segs[active_cols[0]])
for k in range(1, n_y):
    curr_seg = str(r2_segs[active_cols[k]])
    if curr_seg != prev_seg:
        ax1.axhline(y=k - 0.5, color='black', linestyle='-', lw=1.8, zorder=2)
        ax1.text(n_x - 0.3, k - 0.5 + 0.3,
                 f'Chain {curr_seg}',
                 fontsize=7.5, color='black', fontstyle='italic', va='bottom', ha='right')
    prev_seg = curr_seg

ax1.set_xlabel('HVGlu3 residue', fontsize=11, labelpad=8)
ax1.set_ylabel('CESA1 residue',  fontsize=11, labelpad=8)
ax1.tick_params(direction='in', top=True, right=True, labelsize=8)
for spine in ax1.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(0.8)

# Panel label
ax1.text(-0.08, 1.02, '(a)', transform=ax1.transAxes,
         fontsize=14, fontweight='bold', va='bottom')

# ══════════════════════════════════════════════
# Fig. 1b — Top contact pairs bar chart
# ══════════════════════════════════════════════
ax2.set_facecolor('white')

bar_colors = plt.cm.YlOrRd(pair_freqs)

bars = ax2.barh(
    np.arange(len(pair_labels)),
    pair_freqs,
    color=bar_colors,
    edgecolor='#333333',
    linewidth=0.5,
    height=0.72
)

# Frequency values at bar ends
for i, (bar, fv) in enumerate(zip(bars, pair_freqs)):
    ax2.text(fv + 0.012, i, f'{fv:.2f}',
             va='center', ha='left', fontsize=7.5, fontweight='bold', color='#222222')

# Reference lines
ax2.axvline(x=0.5, color='#888888', linestyle='--', lw=0.9, alpha=0.7, label='50%')
ax2.axvline(x=0.7, color='#555555', linestyle=':',  lw=0.9, alpha=0.7, label='70%')
ax2.legend(fontsize=8, loc='lower right', framealpha=0.8)

ax2.set_yticks(np.arange(len(pair_labels)))
ax2.set_yticklabels(pair_labels, fontsize=7.5)
ax2.set_xlabel('Contact frequency', fontsize=11, labelpad=8)
ax2.set_xlim(0, 1.18)
ax2.tick_params(direction='in', top=True, right=True, labelsize=9)
for spine in ax2.spines.values():
    spine.set_edgecolor('black')
    spine.set_linewidth(0.8)

# Panel label
ax2.text(-0.18, 1.02, '(b)', transform=ax2.transAxes,
         fontsize=14, fontweight='bold', va='bottom')

# ── Shared title ──
fig.suptitle(
    f'HVGlu3 ↔ CESA1 Interfacial Contact Frequencies  (cutoff {CUTOFF} Å, freq ≥ {int(THRESHOLD*100)}%)',
    fontsize=13, fontweight='bold', y=0.98
)

# ── Save ──
out_path = OUT_DIR / 'Fig1_contact_map_overall.png'
fig.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'✅ Saved: {out_path}')

In [ ]:
# ============================================================
# CELL — Fig. 6 replacement: Residue × Cluster contact heatmap
# Rows = top HvGLU3 residues (by max frequency across clusters)
# Columns = Cluster 1-4
# Color = contact frequency (max per residue per cluster)
# ============================================================

THRESHOLD_DISPLAY = 0.30   # minimum frequency in at least one cluster to be shown
TOP_N_RESIDUES     = 25    # cap rows for readability; set None for all

# ── Build per-cluster max frequency per HvGLU3 residue ──
residue_freq_by_cluster = {}
all_resids = set()

for cluster_num, df in cluster_contact_dfs.items():
    max_per_resid = df.groupby(['HVGlu3_resid', 'HVGlu3_resname'])['frequency'].max()
    residue_freq_by_cluster[cluster_num] = max_per_resid
    all_resids.update(df['HVGlu3_resid'].unique())

# ── Keep only residues that hit threshold in at least one cluster ──
qualifying_resids = set()
for cluster_num, series in residue_freq_by_cluster.items():
    qualifying_resids.update(series[series >= THRESHOLD_DISPLAY].index.get_level_values(0))

# ── Build the matrix ──
rows = []
for resid in sorted(qualifying_resids):
    row = {'resid': resid}
    resname = None
    overall_max = 0
    for cluster_num in [1, 2, 3, 4]:
        series = residue_freq_by_cluster[cluster_num]
        match = series[series.index.get_level_values(0) == resid]
        if len(match):
            freq = match.values[0]
            resname = match.index.get_level_values(1)[0]
        else:
            freq = 0.0
        row[f'C{cluster_num}'] = freq
        overall_max = max(overall_max, freq)
    row['resname'] = resname
    row['overall_max'] = overall_max
    rows.append(row)

df_heatmap = pd.DataFrame(rows).sort_values('overall_max', ascending=False)

if TOP_N_RESIDUES:
    df_heatmap = df_heatmap.head(TOP_N_RESIDUES)

df_heatmap = df_heatmap.sort_values('resid')  # sort by residue number for readability

print(f'Showing {len(df_heatmap)} residues in heatmap')
print(df_heatmap[['resid','resname','C1','C2','C3','C4']].to_string(index=False))

# ── Plot ──
from matplotlib import rcParams
rcParams['font.family']  = 'Arial'
rcParams['pdf.fonttype'] = 42

fig, ax = plt.subplots(figsize=(6, max(8, len(df_heatmap)*0.32)))

matrix = df_heatmap[['C1','C2','C3','C4']].values
labels = [f"{r['resname']}{r['resid']}" for _, r in df_heatmap.iterrows()]

im = ax.imshow(matrix, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)

ax.set_xticks(range(4))
ax.set_xticklabels(['Cluster 1\n(24.5%)', 'Cluster 2\n(27.9%)',
                     'Cluster 3\n(12.8%)', 'Cluster 4\n(34.8%)'], fontsize=9)
ax.set_yticks(range(len(labels)))
ax.set_yticklabels(labels, fontsize=8)

# Annotate cells with frequency value
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        val = matrix[i, j]
        color = 'white' if val > 0.6 else 'black'
        ax.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=7, color=color)

ax.set_xlabel('Conformational cluster', fontsize=11, labelpad=10)
ax.set_ylabel('HvGLU3 residue', fontsize=11, labelpad=10)

cbar = plt.colorbar(im, ax=ax, fraction=0.04, pad=0.03)
cbar.set_label('Max contact frequency', fontsize=10)


for spine in ax.spines.values():
    spine.set_visible(False)

plt.tight_layout()
out_path = OUT_DIR / 'Fig6_residue_cluster_heatmap.png'
fig.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(OUT_DIR / 'Fig6_residue_cluster_heatmap.pdf', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'\n✅ Saved: {out_path}')

In [ ]:
# ============================================================
# CELL — Combined Fig: (A) Overall dot matrix ≥50% + (B) Cluster heatmap
# ============================================================

from matplotlib import rcParams
import matplotlib.gridspec as gridspec

rcParams['font.family']  = 'Arial'
rcParams['pdf.fonttype'] = 42

THRESHOLD_A = 0.5   # overall dot matrix threshold, matches text "48 pairs >50%"
THRESHOLD_B = 0.3   # minimum to appear in heatmap (any cluster)
TOP_N_RESIDUES_B = 25

# ── Panel A data: overall ≥50% ──
r1_ids_a   = np.asarray(r1_ids)
r2_ids_a   = np.asarray(r2_ids)
r1_names_a = np.asarray(r1_names)
r2_names_a = np.asarray(r2_names)
r2_segs_a  = np.asarray(r2_segs)

mask_a = cf_overall >= THRESHOLD_A
row_idx, col_idx = np.where(mask_a)
active_rows = np.unique(row_idx)
active_cols = np.unique(col_idx)
n_x, n_y = len(active_rows), len(active_cols)
row_to_pos = {r: i for i, r in enumerate(active_rows)}
col_to_pos = {c: i for i, c in enumerate(active_cols)}
x_pos = np.array([row_to_pos[r] for r in row_idx])
y_pos = np.array([col_to_pos[c] for c in col_idx])
freq_a = cf_overall[row_idx, col_idx]
x_labels_a = [f'{r1_names_a[i]}{r1_ids_a[i]}' for i in active_rows]
y_labels_a = [f'{r2_names_a[j]}{r2_ids_a[j]} ({r2_segs_a[j]})' for j in active_cols]

# ── Panel B data: residue x cluster heatmap (from previous cell) ──
residue_freq_by_cluster = {}
for cluster_num, df in cluster_contact_dfs.items():
    residue_freq_by_cluster[cluster_num] = df.groupby(
        ['HVGlu3_resid', 'HVGlu3_resname'])['frequency'].max()

qualifying_resids = set()
for series in residue_freq_by_cluster.values():
    qualifying_resids.update(series[series >= THRESHOLD_B].index.get_level_values(0))

rows = []
for resid in sorted(qualifying_resids):
    row = {'resid': resid}
    resname, overall_max = None, 0
    for cluster_num in [1, 2, 3, 4]:
        series = residue_freq_by_cluster[cluster_num]
        match = series[series.index.get_level_values(0) == resid]
        freq = match.values[0] if len(match) else 0.0
        if len(match):
            resname = match.index.get_level_values(1)[0]
        row[f'C{cluster_num}'] = freq
        overall_max = max(overall_max, freq)
    row['resname'] = resname
    row['overall_max'] = overall_max
    rows.append(row)

df_heatmap = pd.DataFrame(rows).sort_values('overall_max', ascending=False)
if TOP_N_RESIDUES_B:
    df_heatmap = df_heatmap.head(TOP_N_RESIDUES_B)
df_heatmap = df_heatmap.sort_values('resid')

matrix_b = df_heatmap[['C1','C2','C3','C4']].values
labels_b = [f"{r['resname']}{r['resid']}" for _, r in df_heatmap.iterrows()]

# ── Figure layout ──
fig = plt.figure(figsize=(18, max(9, max(n_y, len(labels_b)) * 0.3 + 2)))
gs = gridspec.GridSpec(1, 2, width_ratios=[1.6, 1], wspace=0.5,
                        left=0.07, right=0.95, top=0.92, bottom=0.15)

axA = fig.add_subplot(gs[0])
axB = fig.add_subplot(gs[1])

# ══ Panel A — Overall dot matrix ══
for i in range(n_x):
    axA.axvline(x=i, color='#eeeeee', linewidth=0.4, zorder=0)
for j in range(n_y):
    axA.axhline(y=j, color='#eeeeee', linewidth=0.4, zorder=0)

scA = axA.scatter(x_pos, y_pos, c=freq_a, s=70, cmap='YlOrRd',
                   vmin=THRESHOLD_A, vmax=1.0, edgecolors='#444444',
                   linewidths=0.5, zorder=3)

cbarA = plt.colorbar(scA, ax=axA, fraction=0.025, pad=0.02)
cbarA.set_label('Contact frequency', fontsize=10)

axA.set_xticks(np.arange(n_x))
axA.set_xticklabels(x_labels_a, fontsize=7, rotation=90, ha='center')
axA.set_yticks(np.arange(n_y))
axA.set_yticklabels(y_labels_a, fontsize=7.5)
axA.set_xlim(-0.5, n_x - 0.5)
axA.set_ylim(-0.5, n_y - 0.5)

prev_seg = str(r2_segs_a[active_cols[0]])
for k in range(1, n_y):
    curr_seg = str(r2_segs_a[active_cols[k]])
    if curr_seg != prev_seg:
        axA.axhline(y=k - 0.5, color='black', linestyle='-', lw=1.8, zorder=2)
        axA.text(n_x - 0.3, k - 0.5 + 0.3, f'Chain {curr_seg}',
                  fontsize=7.5, color='black', fontstyle='italic', va='bottom', ha='right')
    prev_seg = curr_seg

axA.set_xlabel('HvGLU3 residue', fontsize=11, labelpad=8)
axA.set_ylabel('CESA1 residue', fontsize=11, labelpad=8)
axA.tick_params(direction='in', top=True, right=True, labelsize=8)
for spine in axA.spines.values():
    spine.set_edgecolor('black'); spine.set_linewidth(0.8)
axA.text(-0.1, 1.02, '(a)', transform=axA.transAxes, fontsize=15, fontweight='bold', va='bottom')

# ══ Panel B — Cluster heatmap ══
imB = axB.imshow(matrix_b, aspect='auto', cmap='YlOrRd', vmin=0, vmax=1)

axB.set_xticks(range(4))
axB.set_xticklabels(['Cluster 1\n(24.5%)', 'Cluster 2\n(27.9%)',
                      'Cluster 3\n(12.8%)', 'Cluster 4\n(34.8%)'], fontsize=9)
axB.set_yticks(range(len(labels_b)))
axB.set_yticklabels(labels_b, fontsize=8)

for i in range(matrix_b.shape[0]):
    for j in range(matrix_b.shape[1]):
        val = matrix_b[i, j]
        color = 'white' if val > 0.6 else 'black'
        axB.text(j, i, f'{val:.2f}', ha='center', va='center', fontsize=7, color=color)

axB.set_xlabel('Conformational cluster', fontsize=11, labelpad=8)
axB.set_ylabel('HvGLU3 residue', fontsize=11, labelpad=8)
cbarB = plt.colorbar(imB, ax=axB, fraction=0.04, pad=0.03)
cbarB.set_label('Max contact frequency', fontsize=10)
for spine in axB.spines.values():
    spine.set_visible(False)
axB.text(-0.15, 1.02, '(b)', transform=axB.transAxes, fontsize=15, fontweight='bold', va='bottom')

# ── Save ──
out_path = OUT_DIR / 'Fig_combined_overall_and_clusters.png'
fig.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
fig.savefig(OUT_DIR / 'Fig_combined_overall_and_clusters.pdf', dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'✅ Saved: {out_path}')

In [ ]:
# ============================================================
# CELL 12 — Headline numbers for paper
# ============================================================

print('=== HEADLINE NUMBERS FOR PAPER ===')
print(f'\nCutoff used: {CUTOFF} Å')
print(f'Total frames analyzed: {len(u.trajectory)}')
print(f'\nOverall contact map (all frames):')
print(f'  Contacts > 30%: {(cf_overall > 0.3).sum()}')
print(f'  Contacts > 50%: {(cf_overall > 0.5).sum()}')
print(f'  Contacts > 70%: {(cf_overall > 0.7).sum()}')
print(f'  Max frequency : {cf_overall.max():.3f}')

for cn in [1, 2, 3, 4]:
    cf_c = cf_clusters[cn]['cf']
    pop  = cluster_populations[cn]
    print(f'\nCluster {cn} ({pop}%):')
    print(f'  Contacts > 30%: {(cf_c > 0.3).sum()}')
    print(f'  Contacts > 50%: {(cf_c > 0.5).sum()}')
    print(f'  Max frequency : {cf_c.max():.3f}')

print('\nAll output files saved to:', OUT_DIR)

In [ ]:
import sys
!{sys.executable} -m pip install py3Dmol

In [ ]:
# ============================================================
# CELL — 3D Visualization of Cluster 3 Key Binding Residues
# Install py3Dmol if needed: pip install py3Dmol
# ============================================================

import py3Dmol
import numpy as np

# ── Cluster 3 key residues (freq > 50%) from your results ──

# HVGlu3 (Chain A)
hvglu3_key = [63, 64, 65, 57, 59, 60, 61, 62, 69, 70, 72, 73, 74, 77, 78, 81]

# CESA1 subunit 1 (Chain B)
cesa1_B_key = [282, 285, 286, 883, 884, 886, 887, 889, 890]

# CESA1 subunit 2 (Chain C)
cesa1_C_key = [1785, 1846, 1849, 1850, 1853]

# CESA1 subunit 3 (Chain D) — Cluster 3 specific
cesa1_D_key = []  # add if you identify key D residues from your CSV

# ── Load PDB (first frame is fine for visualization) ──
pdb_path = str(TOPOLOGY)   # uses the path you already defined

with open(pdb_path, 'r') as f:
    pdb_data = f.read()

# ── Build viewer ──
view = py3Dmol.view(width=900, height=600)
view.addModel(pdb_data, 'pdb')

# Full complex — light cartoon background
view.setStyle({'chain': 'A'}, {'cartoon': {'color': '#5b9bd5', 'opacity': 0.5}})  # HVGlu3 — blue
view.setStyle({'chain': 'B'}, {'cartoon': {'color': '#a8d08d', 'opacity': 0.4}})  # CESA1 sub1 — green
view.setStyle({'chain': 'C'}, {'cartoon': {'color': '#f4b183', 'opacity': 0.4}})  # CESA1 sub2 — orange
view.setStyle({'chain': 'D'}, {'cartoon': {'color': '#c9a0dc', 'opacity': 0.4}})  # CESA1 sub3 — purple

# ── Highlight key residues as sticks ──

# HVGlu3 key residues — bright blue sticks + spheres
for res in hvglu3_key:
    view.addStyle(
        {'chain': 'A', 'resi': res},
        {'stick': {'color': '#1a6fbf', 'radius': 0.25},
         'sphere': {'color': '#1a6fbf', 'radius': 0.5}}
    )

# CESA1 sub1 key residues — green
for res in cesa1_B_key:
    view.addStyle(
        {'chain': 'B', 'resi': res},
        {'stick': {'color': '#2d7d32', 'radius': 0.25},
         'sphere': {'color': '#2d7d32', 'radius': 0.5}}
    )

# CESA1 sub2 key residues — orange
for res in cesa1_C_key:
    view.addStyle(
        {'chain': 'C', 'resi': res},
        {'stick': {'color': '#e65100', 'radius': 0.25},
         'sphere': {'color': '#e65100', 'radius': 0.5}}
    )

# ── Add residue labels for the top contacts ──
top_contacts = {
    ('A', 74): 'LEU74',  ('A', 77): 'VAL77',  ('A', 73): 'MET73',
    ('A', 63): 'VAL63',  ('A', 65): 'VAL65',
    ('B', 282): 'PHE282', ('B', 286): 'TYR286', ('B', 886): 'MET886',
    ('B', 887): 'PHE887', ('B', 889): 'ILE889',
    ('C', 1846): 'SER1846', ('C', 1849): 'SER1849',
}

for (chain, res), label in top_contacts.items():
    view.addResLabels(
        {'chain': chain, 'resi': res},
        {'font': 'Arial', 'fontSize': 10,
         'fontColor': 'black', 'backgroundOpacity': 0.6,
         'backgroundColor': 'white'}
    )

# ── Zoom into the binding interface ──
# Zoom on HVGlu3 (Chain A) — the interface is around it
view.zoomTo({'chain': 'A'})

view.setBackgroundColor('white')
view.show()

In [ ]:
# ============================================================
# CELL — Publication-quality PNG of Cluster 3 key residues
# Uses only MDAnalysis + matplotlib (already installed)
# ============================================================

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.colors import to_rgba

# ── Key residues from Cluster 3 ──
hvglu3_key  = [57, 59, 60, 61, 62, 63, 64, 65, 69, 70, 72, 73, 74, 77, 78, 81]
cesa1_B_key = [282, 285, 286, 883, 884, 886, 887, 889, 890]
cesa1_C_key = [1785, 1846, 1849, 1850, 1853]

# Labels for the very top contacts only (to avoid clutter)
top_labels = {
    ('A', 74): 'LEU74', ('A', 77): 'VAL77', ('A', 73): 'MET73',
    ('A', 63): 'VAL63', ('A', 65): 'VAL65',
    ('B', 282): 'PHE282', ('B', 286): 'TYR286',
    ('B', 886): 'MET886', ('B', 887): 'PHE887',
    ('C', 1846): 'SER1846', ('C', 1849): 'SER1849',
}

# ── Load a representative frame (use middle frame of Cluster 3) ──
# cluster_frames[2] = Cluster 3 indices (0-based key)
c3_frames   = cluster_frames[2]
mid_frame   = c3_frames[len(c3_frames) // 2]
u.trajectory[mid_frame]

# ── Get CA (alpha carbon) positions per chain ──
def get_ca(segid):
    return u.select_atoms(f'segid {segid} and name CA').positions.copy()

ca_A = get_ca('A')
ca_B = get_ca('B')
ca_C = get_ca('C')
ca_D = get_ca('D')

# ── Get CA positions for key residues ──
def get_key_ca(segid, resid_list):
    sel = u.select_atoms(
        f'segid {segid} and name CA and resid {" ".join(map(str, resid_list))}'
    )
    return sel.positions.copy(), [r.resid for r in sel.residues]

kA_pos, kA_ids = get_key_ca('A', hvglu3_key)
kB_pos, kB_ids = get_key_ca('B', cesa1_B_key)
kC_pos, kC_ids = get_key_ca('C', cesa1_C_key)

# ── Plot: 3 projection views ──
views = [
    ('Front view',  0, 1, 'X (Å)', 'Y (Å)'),
    ('Side view',   2, 1, 'Z (Å)', 'Y (Å)'),
    ('Top view',    0, 2, 'X (Å)', 'Z (Å)'),
]

fig, axes = plt.subplots(1, 3, figsize=(22, 8))
fig.patch.set_facecolor('black')

for ax, (title, xi, yi, xlabel, ylabel) in zip(axes, views):
    ax.set_facecolor('#0d0d0d')

    # ── Full chain backbones (lines) ──
    ax.plot(ca_A[:, xi], ca_A[:, yi], '-', color='#5b9bd5', lw=0.8, alpha=0.5, label='HVGlu3')
    ax.plot(ca_B[:, xi], ca_B[:, yi], '-', color='#57a85a', lw=0.8, alpha=0.4, label='CESA1 sub1')
    ax.plot(ca_C[:, xi], ca_C[:, yi], '-', color='#e07b39', lw=0.8, alpha=0.4, label='CESA1 sub2')
    ax.plot(ca_D[:, xi], ca_D[:, yi], '-', color='#b07cc6', lw=0.8, alpha=0.4, label='CESA1 sub3')

    # ── Key residues (bright spheres) ──
    ax.scatter(kA_pos[:, xi], kA_pos[:, yi],
               s=80, color='#00cfff', zorder=5, edgecolors='white', linewidths=0.5)
    ax.scatter(kB_pos[:, xi], kB_pos[:, yi],
               s=80, color='#00ff99', zorder=5, edgecolors='white', linewidths=0.5)
    ax.scatter(kC_pos[:, xi], kC_pos[:, yi],
               s=80, color='#ffaa00', zorder=5, edgecolors='white', linewidths=0.5)

    # ── Labels for top contacts ──
    for (chain, resid), name in top_labels.items():
        if chain == 'A':
            pos, color = kA_pos, '#00cfff'
            ids = kA_ids
        elif chain == 'B':
            pos, color = kB_pos, '#00ff99'
            ids = kB_ids
        else:
            pos, color = kC_pos, '#ffaa00'
            ids = kC_ids

        if resid in ids:
            idx = list(ids).index(resid)
            ax.annotate(
                name,
                xy=(pos[idx, xi], pos[idx, yi]),
                xytext=(8, 8), textcoords='offset points',
                fontsize=7, color=color, fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', fc='black', alpha=0.6, ec='none')
            )

    ax.set_title(title, color='white', fontsize=13, fontweight='bold', pad=10)
    ax.set_xlabel(xlabel, color='#aaaaaa', fontsize=10)
    ax.set_ylabel(ylabel, color='#aaaaaa', fontsize=10)
    ax.tick_params(colors='#666666', labelsize=8)
    for spine in ax.spines.values():
        spine.set_edgecolor('#333333')

# ── Legend ──
legend_elements = [
    mpatches.Patch(color='#5b9bd5', label='HVGlu3 (Chain A)'),
    mpatches.Patch(color='#57a85a', label='CESA1 subunit 1 (B)'),
    mpatches.Patch(color='#e07b39', label='CESA1 subunit 2 (C)'),
    mpatches.Patch(color='#b07cc6', label='CESA1 subunit 3 (D)'),
    plt.scatter([], [], s=80, color='#00cfff', edgecolors='white', linewidths=0.5, label='HVGlu3 key residues'),
    plt.scatter([], [], s=80, color='#00ff99', edgecolors='white', linewidths=0.5, label='CESA1 sub1 key residues'),
    plt.scatter([], [], s=80, color='#ffaa00', edgecolors='white', linewidths=0.5, label='CESA1 sub2 key residues'),
]

fig.legend(
    handles=legend_elements,
    loc='lower center', ncol=4,
    fontsize=9, framealpha=0.15,
    facecolor='#1a1a1a', edgecolor='#444444',
    labelcolor='white',
    bbox_to_anchor=(0.5, -0.05)
)

fig.suptitle(
    'Cluster 3 — Key Binding Residues\nHVGlu3 ↔ CESA1 Interface (Cα representation)',
    color='white', fontsize=15, fontweight='bold', y=1.02
)

plt.tight_layout()
out_path = OUT_DIR / 'cluster3_key_residues_3views.png'
fig.savefig(out_path, dpi=300, bbox_inches='tight',
            facecolor='black', edgecolor='none')
plt.show()
print(f'✅ Saved: {out_path}')

In [ ]:
# ============================================================
# CELL — Extract global variables from cf_overall
# Run this before the Fig. 1 cell
# ============================================================

sel1 = u.select_atoms('segid A')
sel2 = u.select_atoms('segid B or segid C')

res1 = sel1.residues
res2 = sel2.residues

r1_ids   = res1.resids
r2_ids   = res2.resids
r1_names = np.array([r.resname for r in res1])
r2_names = np.array([r.resname for r in res2])
r1_segs  = np.array([r.segid   for r in res1])
r2_segs  = np.array([r.segid   for r in res2])

print(f'r1 (HVGlu3):  {len(r1_ids)} residues')
print(f'r2 (CESA1):   {len(r2_ids)} residues')
print(f'cf_overall shape: {cf_overall.shape}')
print('✅ Ready for Fig. 1')

In [ ]:
# ============================================================
# CELL — 2D Interaction Diagram — Cluster 3
# HvGlu3 ↔ CESA1 binding interface
# ============================================================

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from matplotlib.lines import Line2D
import numpy as np

# ── Load Cluster 3 contact data ──
data = """HVGlu3_resid,HVGlu3_resname,CESA1_chain,CESA1_resid,CESA1_resname,frequency
63,VAL,B,889,ILE,1.0
74,LEU,B,282,PHE,1.0
65,VAL,B,886,MET,1.0
77,VAL,B,286,TYR,0.999
77,VAL,B,282,PHE,0.997
57,TYR,C,1850,LEU,0.992
64,VAL,C,1849,SER,0.992
73,MET,B,886,MET,0.991
73,MET,B,887,PHE,0.987
81,PHE,B,286,TYR,0.981
59,ASP,C,1846,SER,0.981
73,MET,B,884,ALA,0.981
61,GLY,C,1846,SER,0.979
70,LEU,B,886,MET,0.979
62,CYS,C,1785,VAL,0.973
60,LEU,B,890,LEU,0.952
77,VAL,B,285,GLN,0.948
63,VAL,B,890,LEU,0.943
69,LEU,B,884,ALA,0.937
69,LEU,B,883,TYR,0.937
63,VAL,B,893,ALA,0.92
62,CYS,C,1842,ILE,0.915
81,PHE,B,290,HSD,0.911
62,CYS,C,1846,SER,0.909
73,MET,B,880,ILE,0.902
77,VAL,B,887,PHE,0.902
62,CYS,B,893,ALA,0.901
88,ILE,B,881,SER,0.897
73,MET,B,282,PHE,0.892
84,ILE,B,880,ILE,0.876
62,CYS,B,889,ILE,0.873
61,GLY,B,893,ALA,0.871
64,VAL,C,1850,LEU,0.87"""

from io import StringIO
df = pd.read_csv(StringIO(data))

# ── Filter to frequency ≥ 0.85 for clarity ──
df = df[df['frequency'] >= 0.85].copy()

# ── Create residue labels ──
df['hvglu3_label'] = df['HVGlu3_resname'] + df['HVGlu3_resid'].astype(str)
df['cesa1_label']  = df['CESA1_resname'] + df['CESA1_resid'].astype(str) + \
                     ' (' + df['CESA1_chain'] + ')'

# ── Unique residues ──
hvglu3_res = sorted(df['hvglu3_label'].unique(),
                    key=lambda x: int(''.join(filter(str.isdigit, x))))
cesa1_res  = sorted(df['cesa1_label'].unique(),
                    key=lambda x: int(''.join(filter(str.isdigit, x.split('(')[0]))))

# ── Position mapping ──
hvglu3_pos = {r: i for i, r in enumerate(hvglu3_res)}
cesa1_pos  = {r: i for i, r in enumerate(cesa1_res)}

# ── Chain colors for CESA1 nodes ──
chain_colors = {'B': '#2E86AB', 'C': '#E07B39'}

# ── Frequency colormap ──
cmap = plt.cm.YlOrRd

# ── Figure ──
fig, ax = plt.subplots(figsize=(14, max(10, max(len(hvglu3_res),
                                              len(cesa1_res)) * 0.6)))
fig.patch.set_facecolor('white')
ax.set_facecolor('white')
ax.set_xlim(-1, 4)
ax.set_ylim(-1, max(len(hvglu3_res), len(cesa1_res)))
ax.axis('off')

# ── Draw connection lines ──
for _, row in df.iterrows():
    hl = row['hvglu3_label']
    cl = row['cesa1_label']
    y1 = hvglu3_pos[hl]
    y2 = cesa1_pos[cl]
    freq = row['frequency']
    color = cmap(freq)
    lw = 1.5 + freq * 2.5

    ax.plot([0.35, 2.65], [y1, y2],
            color=color, lw=lw, alpha=0.7, zorder=1)

# ── Draw HvGlu3 nodes (left) ──
for res, y in hvglu3_pos.items():
    ax.scatter(0.3, y, s=400, color='#C0392B',
               zorder=3, edgecolors='white', linewidths=1.5)
    ax.text(0.2, y, res, ha='right', va='center',
            fontsize=10, fontweight='bold', color='#C0392B')

# ── Draw CESA1 nodes (right) ──
for res, y in cesa1_pos.items():
    chain = res.split('(')[1].replace(')', '')
    color = chain_colors.get(chain, '#888888')
    ax.scatter(2.7, y, s=400, color=color,
               zorder=3, edgecolors='white', linewidths=1.5)
    ax.text(2.8, y, res, ha='left', va='center',
            fontsize=10, fontweight='bold', color=color)

# ── Column headers ──
ax.text(0.3, len(hvglu3_res) + 0.3, 'HvGlu3',
        ha='center', va='bottom', fontsize=13,
        fontweight='bold', color='#C0392B')
ax.text(2.7, max(len(hvglu3_res), len(cesa1_res)) + 0.3, 'CESA1',
        ha='center', va='bottom', fontsize=13,
        fontweight='bold', color='#2E86AB')

# ── Colorbar ──
sm = plt.cm.ScalarMappable(cmap=cmap,
                            norm=plt.Normalize(vmin=0.85, vmax=1.0))
sm.set_array([])
cbar = plt.colorbar(sm, ax=ax, fraction=0.02, pad=0.01,
                    shrink=0.4, anchor=(0.0, 0.3))
cbar.set_label('Contact frequency', fontsize=10)
cbar.ax.tick_params(labelsize=9)

# ── Legend ──
legend_elements = [
    mpatches.Patch(color='#C0392B', label='HvGlu3'),
    mpatches.Patch(color='#2E86AB', label='CESA1 subunit 1 (B)'),
    mpatches.Patch(color='#E07B39', label='CESA1 subunit 2 (C)'),
]
ax.legend(handles=legend_elements, loc='lower right',
          fontsize=10, framealpha=1.0,
          facecolor='white', edgecolor='black')

ax.set_title('HvGlu3 ↔ CESA1 Binding Interface — Cluster 3\n'
             'Residue contact diagram (frequency ≥ 85%)',
             fontsize=13, fontweight='bold', pad=15)

plt.tight_layout()
out_path = OUT_DIR / 'Fig_2D_interaction_cluster3.png'
fig.savefig(out_path, dpi=300, bbox_inches='tight', facecolor='white')
plt.show()
print(f'✅ Saved: {out_path}')

In [ ]:
import pandas as pd
from pathlib import Path

OUT_DIR = Path('/Volumes/SAMSUNG/CESAproject/MD_after_docking/CESA1/result_CESA1_ClusPro/run/ContactMap')

df_c3 = pd.read_csv(OUT_DIR / 'key_contacts_cluster3.csv')
print(f'Total Cluster 3 contacts ≥50%: {len(df_c3)}')
print(df_c3['CESA1_chain'].value_counts())

df_c3_chainD = df_c3[df_c3['CESA1_chain'] == 'D'].sort_values('frequency', ascending=False)
print(f'\nChain D contacts ≥50% in Cluster 3: {len(df_c3_chainD)}')
print(df_c3_chainD.to_string(index=False))